#Test

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn grapheme


In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "/content/best_model_10_grapheme"   # folder you saved

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
import pandas as pd

test_df = pd.read_csv("test.csv")   # change name if different
test_df.head()


,Text
0,லக்ஷ்மி அம்மா நீங்க புலம்புங்க அவளுக அவளுகபாட்...
1,"இன்னும் கைது பண்ணல... அனைத்து பெற்றோர்களும், க..."
2,"அப்பா,அம்மா, அந்த இன்டர்வியூ பண்ற வக்கிரம்புடி..."
3,Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க கார...
4,எல்லாமே script thaan. ஷகீலா உங்க scriptum அரும...


In [ ]:
import re, html, unicodedata, grapheme

def grapheme_preprocess(text):
    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r'http\S+|www\S+', '', text)
    clusters = list(grapheme.graphemes(text))
    text = "".join(clusters)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

test_df["processed"] = test_df["Text"].apply(grapheme_preprocess)


In [ ]:
predictions = []

for text in test_df["processed"]:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    predictions.append(pred)


In [ ]:
label_map = {0:"Non-Abusive", 1:"Abusive"}
test_df["Prediction"] = [label_map[p] for p in predictions]


In [ ]:
submission = test_df[["Text","Prediction"]]
submission.to_csv("submission.csv", index=False)


In [ ]:
print(test_df.head())
print(test_df.columns)
